# SanteScope — 05_export_json.ipynb

Export toutes les donnees vers des fichiers JSON statiques pour le frontend Next.js.
- `public/data/index.json` : liste legere pour la recherche (~35K entrees)
- `public/data/communes/{code}.json` : fiche complete par commune

**Phase 02, Plan 02, Task 2** — Schema valide par jsonschema.

---

In [1]:
import pandas as pd
import numpy as np
import json
import pathlib
import jsonschema
import time
import os

# Path-agnostic detection
if pathlib.Path('notebooks/data/processed').exists():
    PROCESSED = pathlib.Path('notebooks/data/processed')
    PROJECT_ROOT = pathlib.Path('.').resolve()
elif pathlib.Path('data/processed').exists():
    PROCESSED = pathlib.Path('data/processed')
    PROJECT_ROOT = pathlib.Path('..').resolve()
else:
    raise FileNotFoundError('Cannot find data/processed directory')

df = pd.read_parquet(PROCESSED / 'communes_master.parquet')
medians = json.load(open(PROCESSED / 'national_medians.json'))

OUTPUT_DIR = PROJECT_ROOT / 'public' / 'data'
COMMUNES_DIR = OUTPUT_DIR / 'communes'
COMMUNES_DIR.mkdir(parents=True, exist_ok=True)

assert (PROJECT_ROOT / 'notebooks').exists(), f'Wrong project root: {PROJECT_ROOT}'

print(f'Loaded: {df.shape[0]} communes, {df.shape[1]} columns')
print(f'Medians: {medians}')
print(f'Output: {OUTPUT_DIR}')


Loaded: 34969 communes, 46 columns
Medians: {'apl': 2.88, 'revenu_median': 21450.0, 'pct_75_seuls': 0.0922, 'temps_urgences_min': 22.0, 'pauvrete': 0.12}
Output: /Users/zakariyahamdouchi/Dev/Hackathon/public/data


## Section 2 — Classification qualite donnees

- **complete** : 4/4 composantes du score
- **partial** : 3/4 composantes
- **minimal** : <3 composantes (score = null)

In [2]:
def data_quality(row):
    n = row['n_score_components']
    if n >= 4:
        return 'complete'
    elif n >= 3:
        return 'partial'
    else:
        return 'minimal'

df['data_quality'] = df.apply(data_quality, axis=1)
print(df['data_quality'].value_counts())


data_quality
complete    31186
partial      3639
minimal       144
Name: count, dtype: int64


## Section 3 — JSON Schema

Schema conforme au contrat gele Phase 1 avec adaptations documentees:
- `apl_evolution`: 2 ans (2022-2023) au lieu de 3
- `classe` et `data_quality`: champs ajoutes
- `jumelles[].apl_avant/apl_apres` au lieu de `score_avant/score_apres`

In [3]:
COMMUNE_SCHEMA = {
    "type": "object",
    "required": ["code", "nom", "dept", "pop", "data_quality"],
    "properties": {
        "code": {"type": "string"},
        "nom": {"type": "string"},
        "dept": {"type": "string"},
        "region": {"type": ["string", "null"]},
        "pop": {"type": "integer"},
        "score": {"type": ["number", "null"]},
        "classe": {"type": ["string", "null"]},
        "data_quality": {"type": "string", "enum": ["complete", "partial", "minimal"]},
        "score_detail": {
            "type": "object",
            "properties": {
                "apl": {"type": ["number", "null"]},
                "apl_national": {"type": "number"},
                "pauvrete": {"type": ["number", "null"]},
                "pauvrete_national": {"type": "number"},
                "pct_75_seuls": {"type": ["number", "null"]},
                "pct_75_seuls_national": {"type": "number"},
                "temps_urgences_min": {"type": ["number", "null"]},
                "temps_urgences_national": {"type": "number"}
            }
        },
        "medecins": {
            "type": "object",
            "properties": {
                "generalistes": {"type": "integer"},
                "specialistes": {"type": "object"},
                "total": {"type": "integer"}
            }
        },
        "manques": {"type": ["array", "null"]},
        "domino": {"type": ["object", "null"]},
        "jumelles": {"type": "array"},
        "msp_presente": {"type": "boolean"},
        "apl_evolution": {"type": ["object", "null"]},
        "pathologies_dept": {"type": ["object", "null"]},
        "coords": {"type": ["array", "null"]}
    }
}
print("Schema defined")


Schema defined


## Section 4 — Build Commune Dict

In [ ]:
def to_native(val):
    if isinstance(val, (np.integer,)):
        return int(val)
    if isinstance(val, (np.floating,)):
        return round(float(val), 2) if pd.notna(val) else None
    if isinstance(val, (np.bool_,)):
        return bool(val)
    if isinstance(val, float) and pd.isna(val):
        return None
    if pd.isna(val) if not isinstance(val, (str, list, dict, bool)) else False:
        return None
    return val

def build_commune_dict(row, medians):
    # Parse specialistes_detail
    spec_detail = {}
    sd = row.get('specialistes_detail')
    if sd is not None and not (isinstance(sd, float) and pd.isna(sd)):
        try:
            spec_detail = json.loads(sd) if isinstance(sd, str) else sd
        except:
            pass
    
    # Domino
    domino = None
    da = row.get('domino_alert')
    if da is not None and not (isinstance(da, float) and pd.isna(da)):
        domino = {
            "medecins_55_plus": to_native(row.get('medecins_55plus_est')),
            "pct_55_plus": to_native(row.get('pct_55plus_dept')),
            "pct_55_plus_dept": to_native(row.get('pct_55plus_dept')),
            "projection_2030": to_native(row.get('projection_2030')),
        }
        # Add trend fields if available
        eff_2025 = row.get('effectif_medecins_dept_2025')
        if pd.notna(eff_2025):
            domino["effectif_dept_2025"] = int(eff_2025)
        cagr = row.get('trend_medecins_cagr')
        if pd.notna(cagr):
            domino["trend_cagr"] = round(float(cagr), 2)
        delta = row.get('trend_medecins_delta_annuel')
        if pd.notna(delta):
            domino["trend_delta_annuel"] = round(float(delta), 1)
    
    # APL evolution (all available years 2015-2023)
    apl_evo = None
    _apl_years = [2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023]
    _evo_tmp = {}
    for _yr in _apl_years:
        _col = f'apl_{_yr}'
        if _col in row.index and pd.notna(row[_col]):
            _evo_tmp[str(_yr)] = round(float(row[_col]), 2)
    if _evo_tmp:
        apl_evo = _evo_tmp
    
    # Pathologies dept
    patho = None
    patho_cols = ['prev_diabete', 'prev_cardio', 'prev_psy', 'prev_cancers', 'prev_respiratoire']
    if any(pd.notna(row.get(c)) for c in patho_cols):
        patho = {
            "diabete": to_native(row.get('prev_diabete')),
            "cardiovasculaire": to_native(row.get('prev_cardio')),
            "psychiatrique": to_native(row.get('prev_psy')),
            "cancers": to_native(row.get('prev_cancers')),
            "respiratoire": to_native(row.get('prev_respiratoire')),
        }
    
    # Coords
    coords = None
    if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude')):
        coords = [round(float(row['latitude']), 4), round(float(row['longitude']), 4)]
    
    # Manques - handle numpy arrays from parquet
    manques = row.get('manques')
    if manques is None:
        pass
    elif isinstance(manques, float) and pd.isna(manques):
        manques = None
    elif hasattr(manques, 'tolist'):
        manques = manques.tolist()
    elif not isinstance(manques, list):
        manques = None
    
    # Jumelles - handle numpy arrays from parquet
    jumelles_raw = row.get('jumelles', [])
    if jumelles_raw is None or (isinstance(jumelles_raw, float) and pd.isna(jumelles_raw)):
        jumelles = []
    else:
        jumelles = []
        for t in jumelles_raw:
            td = {}
            for k, v in (t.items() if isinstance(t, dict) else []):
                if hasattr(v, 'tolist'):
                    td[k] = v.tolist()
                elif isinstance(v, (np.integer,)):
                    td[k] = int(v)
                elif isinstance(v, (np.floating,)):
                    td[k] = float(v) if pd.notna(v) else None
                elif isinstance(v, (np.bool_,)):
                    td[k] = bool(v)
                else:
                    td[k] = v
            jumelles.append(td)
    
    pop = row.get('population', 0)
    pop = int(pop) if pd.notna(pop) else 0
    
    return {
        "code": str(row['code_commune']),
        "nom": str(row['nom_commune']),
        "dept": str(row['code_departement']),
        "region": to_native(row.get('region')),
        "pop": pop,
        "score": to_native(row.get('score')),
        "classe": to_native(row.get('classe')),
        "data_quality": row['data_quality'],
        "score_detail": {
            "apl": to_native(row.get('apl')),
            "apl_national": round(float(medians['apl']), 1),
            "pauvrete": to_native(row.get('taux_pauvrete')),
            "pauvrete_national": round(float(medians['pauvrete']), 4),
            "pct_75_seuls": to_native(row.get('pct_75_plus')),
            "pct_75_seuls_national": round(float(medians['pct_75_seuls']), 4),
            "temps_urgences_min": to_native(row.get('temps_urgences_min')),
            "temps_urgences_national": round(float(medians['temps_urgences_min']), 1),
        },
        "medecins": {
            "generalistes": int(row.get('nb_generalistes', 0)),
            "specialistes": spec_detail if spec_detail else {},
            "total": int(row.get('nb_medecins_total', 0)),
        },
        "manques": manques,
        "domino": domino,
        "jumelles": jumelles,
        "msp_presente": bool(row.get('has_msp', False)),
        "apl_evolution": apl_evo,
        "pathologies_dept": patho,
        "coords": coords,
    }

# Test with one row
sample = build_commune_dict(df.iloc[0], medians)
print(json.dumps(sample, ensure_ascii=False, indent=2)[:500])
jsonschema.validate(sample, COMMUNE_SCHEMA)
print("\nSchema validation OK for sample")

## Section 5 — Export per-commune JSON files

Un fichier JSON par commune (>35K fichiers). Validation jsonschema sur chaque fichier.

In [5]:
t0 = time.time()
errors = []

for idx, (_, row) in enumerate(df.iterrows()):
    if idx % 10000 == 0 and idx > 0:
        elapsed = time.time() - t0
        print(f"  {idx}/{len(df)} communes exportees ({elapsed:.1f}s)")
    
    commune_dict = build_commune_dict(row, medians)
    
    try:
        jsonschema.validate(commune_dict, COMMUNE_SCHEMA)
    except jsonschema.ValidationError as e:
        errors.append((row['code_commune'], str(e.message)[:100]))
        if len(errors) > 100:
            print(f"STOP: too many errors ({len(errors)})")
            break
    
    fpath = COMMUNES_DIR / f"{row['code_commune']}.json"
    with open(fpath, 'w', encoding='utf-8') as f:
        json.dump(commune_dict, f, ensure_ascii=False, separators=(',', ':'))

elapsed = time.time() - t0
print(f"Export complete: {len(df)} communes in {elapsed:.1f}s")

if errors:
    print(f"\nERRORS ({len(errors)}):")
    for code, msg in errors[:10]:
        print(f"  {code}: {msg}")
assert len(errors) == 0, f"Schema validation errors: {errors[:10]}"
print("All communes pass schema validation")


  10000/34969 communes exportees (43.2s)


  20000/34969 communes exportees (86.1s)


  30000/34969 communes exportees (129.0s)


Export complete: 34969 communes in 150.7s
All communes pass schema validation


## Section 6 — Export index.json

Liste legere pour la recherche frontend: code, nom, dept, score, classe, pop.

In [6]:
index_data = []
for _, row in df.iterrows():
    index_data.append({
        "code": str(row['code_commune']),
        "nom": str(row['nom_commune']),
        "dept": str(row['code_departement']),
        "score": round(float(row['score']), 1) if pd.notna(row['score']) else None,
        "classe": str(row['classe']) if pd.notna(row.get('classe')) else None,
        "pop": int(row['population']) if pd.notna(row['population']) else 0,
    })

with open(OUTPUT_DIR / 'index.json', 'w', encoding='utf-8') as f:
    json.dump(index_data, f, ensure_ascii=False, separators=(',', ':'))

assert len(index_data) > 30000, f"Only {len(index_data)} index entries"
index_size = (OUTPUT_DIR / 'index.json').stat().st_size
print(f"index.json: {index_size / 1024:.0f} KB, {len(index_data)} entries")


index.json: 2911 KB, 34969 entries


## Section 7 — Final Validation

In [7]:
file_count = len(list(COMMUNES_DIR.glob('*.json')))
assert file_count > 30000, f"Only {file_count} JSON files"
assert (OUTPUT_DIR / 'index.json').exists(), "index.json missing"

index_size = (OUTPUT_DIR / 'index.json').stat().st_size
print(f"index.json: {index_size / 1024:.0f} KB")
print(f"Commune JSON files: {file_count}")

# Spot-checks
spot_checks = ['02691', '75056', '59392']
for code in spot_checks:
    fpath = COMMUNES_DIR / f'{code}.json'
    if fpath.exists():
        c = json.load(open(fpath))
        print(f"\n{c['nom']} ({code}):")
        print(f"  score={c['score']}, classe={c['classe']}, quality={c['data_quality']}")
        print(f"  twins={len(c['jumelles'])}, msp={c['msp_presente']}")
        print(f"  domino={c['domino'] is not None}, manques={'oui' if c['manques'] else 'non'}")
    else:
        print(f"\n{code}: FILE NOT FOUND")

# Check a null-score commune
null_scores = df[df['score'].isna()]
if len(null_scores) > 0:
    code = null_scores.iloc[0]['code_commune']
    c = json.load(open(COMMUNES_DIR / f'{code}.json'))
    print(f"\nNull-score: {c['nom']} ({code})")
    print(f"  score={c['score']}, classe={c['classe']}, quality={c['data_quality']}")

print(f"\n=== EXPORT COMPLET: {file_count} fichiers + index.json ===")


index.json: 2911 KB
Commune JSON files: 34969

Saint-Quentin (02691):
  score=4.9, classe=B, quality=complete
  twins=5, msp=True
  domino=True, manques=oui

Paris (75056):
  score=None, classe=None, quality=minimal
  twins=5, msp=False
  domino=True, manques=non

Maubeuge (59392):
  score=5.0, classe=C, quality=complete
  twins=5, msp=True
  domino=True, manques=oui

Null-score: Conques-en-Rouergue (12218)
  score=None, classe=None, quality=minimal

=== EXPORT COMPLET: 34969 fichiers + index.json ===
